# Otter × MuseTalk — lip-sync test

Runs **MuseTalk v1.5** on an otter image/video + an audio clip on a free Colab **T4 GPU**.

**Before running:** Runtime → Change runtime type → Hardware accelerator = **T4 GPU**.

Run cells top-to-bottom. First run installs deps + weights (~5–10 min). See `README.md` in this folder for the otter imagery spec.

In [ ]:
#@title 1) Check the GPU (must show a T4 / similar NVIDIA GPU)
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU'

In [ ]:
#@title 2) Clone MuseTalk
%cd /content
![ -d MuseTalk ] || git clone https://github.com/TMElyralab/MuseTalk
%cd /content/MuseTalk
!ls

In [ ]:
#@title 3) Install dependencies (pinned torch 2.0.1+cu118 + mmlab)
#@markdown This is the fragile step. The pins below are MuseTalk's official ones; do not bump them.
%cd /content/MuseTalk
!pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!pip install -q -r requirements.txt
!pip install -q --no-cache-dir -U openmim
!mim install mmengine
!mim install "mmcv==2.0.1"
!mim install "mmdet==3.1.0"
!mim install "mmpose==1.1.0"
print('\n✓ deps installed (ignore pip resolver warnings as long as the mim installs succeeded)')

In [ ]:
#@title 4) Download model weights (~a few GB)
%cd /content/MuseTalk
!pip install -q -U "huggingface_hub[hf_transfer]"
!apt-get -qq install -y git-lfs >/dev/null && git lfs install
!sh ./download_weights.sh
print('\n✓ weights downloaded'); !find models -maxdepth 2 -type f | head -40

In [ ]:
#@title 5) Upload your otter image/video + an audio clip
#@markdown Upload an otter **.png/.jpg** (still) or **.mp4** (driving video), AND an audio **.wav/.mp3**.
#@markdown (Skip the audio file if you'll generate one with the optional Piper cell below.)
from google.colab import files
import os, shutil
%cd /content/MuseTalk
os.makedirs('otter', exist_ok=True)
up = files.upload()
for name in up:
    shutil.move(name, os.path.join('otter', name))
print('\nUploaded:', os.listdir('otter'))

In [ ]:
#@title 5b) OPTIONAL — generate audio with Piper (same voice as the live pipeline)
TEXT = "Hello friends, this old otter is testing a brand new look at the news desk."  #@param {type:"string"}
%cd /content/MuseTalk
!pip install -q piper-tts
!python -m piper.download_voices en_US-ryan-high --data-dir otter/voices
import subprocess
subprocess.run('python -m piper -m otter/voices/en_US-ryan-high.onnx -f otter/otter_audio.wav',
               input=TEXT.encode(), shell=True)
print('✓ wrote otter/otter_audio.wav')

In [ ]:
#@title 6) Configure + run inference
BBOX_SHIFT = 0  #@param {type:"integer"}
#@markdown `bbox_shift`: positive = more mouth openness, negative = less. Tune if the mouth looks wrong.
import glob, os
%cd /content/MuseTalk
imgs = sorted(glob.glob('otter/*.png')+glob.glob('otter/*.jpg')+glob.glob('otter/*.jpeg')+glob.glob('otter/*.mp4')+glob.glob('otter/*.mov'))
auds = sorted(glob.glob('otter/*.wav')+glob.glob('otter/*.mp3')+glob.glob('otter/*.m4a'))
assert imgs, 'No image/video found in otter/'; assert auds, 'No audio found in otter/ (upload one or run the Piper cell)'
video_path, audio_path = imgs[0], auds[0]
yaml = f'task_0:\n  video_path: "{video_path}"\n  audio_path: "{audio_path}"\n  bbox_shift: {BBOX_SHIFT}\n'
open('configs/inference/test.yaml','w').write(yaml)
print('Using:\n'+yaml)
!sh inference.sh v1.5 normal

In [ ]:
#@title 7) Show the result
import glob, os
from IPython.display import Video, display
vids = sorted(glob.glob('results/**/*.mp4', recursive=True), key=os.path.getmtime)
assert vids, 'No output video — check the inference log above for errors (often face-not-detected on a cartoon).'
print('Newest result:', vids[-1])
display(Video(vids[-1], embed=True, width=360))
from google.colab import files; files.download(vids[-1])

## If it fails / looks bad

- **`face not detected` / crash** → the otter snout isn't reading as a face. Make the face more frontal/human-proportioned with a clear mouth (see `README.md`).
- **Uncanny human mouth pasted on the otter** → expected on a flat cartoon; this is the core risk. Iterate the imagery toward a defined, lipped, slightly-open mouth, or reconsider MuseTalk vs a cartoon-optimized tool.
- **Mouth barely moves / too open** → tune `BBOX_SHIFT` in cell 6 (try -7, +5, +10).
- **First baseline:** upload `assets/mouths/X.png` from the jafo repo to see the starting failure mode before investing in new art.